In [2]:
import pandas as pd
import numpy as np

path = r'D:\Data Analyst\Project\olist-ecommerce-dashboard\data\row\\'

orders    = pd.read_csv(path + 'olist_orders_dataset.csv')
items     = pd.read_csv(path + 'olist_order_items_dataset.csv')
payments  = pd.read_csv(path + 'olist_order_payments_dataset.csv')
reviews   = pd.read_csv(path + 'olist_order_reviews_dataset.csv')
customers = pd.read_csv(path + 'olist_customers_dataset.csv')
sellers   = pd.read_csv(path + 'olist_sellers_dataset.csv')
products  = pd.read_csv(path + 'olist_products_dataset.csv')
trans     = pd.read_csv(path + 'product_category_name_translation.csv')


In [3]:
print(orders.isnull().sum())
print(items.isnull().sum())
print(payments.isnull().sum())
print(reviews.isnull().sum())
print(customers.isnull().sum())
print(sellers.isnull().sum())
print(products.isnull().sum())
print(trans.isnull().sum())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city     

-> Parse all date columns

In [4]:
date_cols = ['order_purchase_timestamp','order_approved_at',
             'order_delivered_carrier_date',
             'order_delivered_customer_date',
             'order_estimated_delivery_date']


In [5]:
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [6]:
print(orders[col])

0       2017-10-18
1       2018-08-13
2       2018-09-04
3       2017-12-15
4       2018-02-26
           ...    
99436   2017-03-28
99437   2018-03-02
99438   2017-09-27
99439   2018-02-15
99440   2018-04-03
Name: order_estimated_delivery_date, Length: 99441, dtype: datetime64[ns]


-> Aggregate items per order (one row per order)

In [7]:
items_agg = items.groupby('order_id').agg(
    revenue        = ('price','sum'),
    freight        = ('freight_value','sum'),
    itemas_count   = ('order_item_id','count'),
    avg_item_price = ('price','mean'),
    seller_id      = ('seller_id','first'),
    product_id     = ('product_id','first')
).reset_index()

In [8]:
items_agg

,order_id,revenue,freight,itemas_count,avg_item_price,seller_id,product_id
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,58.90,48436dade18ac8b2bce089ec2a041202,4244733e06e7ecb4970a6e2683c13e61
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,239.90,dd7ddc04e1b6c2c614352b383efe2d36,e5f2d52b802189ee658865ca93d83a8f
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,199.00,5b51032eddd242adc84c38acab88f23d,c777355d18b72b67abbeef9df44fd0fd
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,12.99,9d7a1d34a5052409006425275ba1c2b4,7634da152a4610f1595efa32f14722fc
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,199.90,df560393f3a51e74553ab94004ba5c87,ac6c3623068f30de03045865e4e10089
...,...,...,...,...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,299.99,43.41,1,299.99,b8bc237ba3788b23da09c0f1f3a3288c,4aa6014eceb682077f9dc4bffebc05b0
98662,fffcd46ef2263f404302a634eb57f7eb,350.00,36.53,1,350.00,f3c38ab652836d21de61fb8314b69182,32e07fd915822b0765e448c4dd74c828
98663,fffce4705a9662cd70adb13d4a31832d,99.90,16.95,1,99.90,c3cfdc648177fdbbbb35635a37472c53,72a30483855e2eafc67aee5dc2560482
98664,fffe18544ffabc95dfada21779c9644f,55.99,8.72,1,55.99,2b3e4a2a3ea8e01938cabda2a3e5cc79,9c422a519119dcad7575db5af1ba540e


-> Aggregate payments per order

In [9]:
pay_agg = payments.groupby('order_id').agg(
    total_payment = ('payment_value','sum'),
    payment_type  = ('payment_type','first'),
    payment_installments = ('payment_installments','max')
).reset_index()

In [10]:
payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [11]:
pay_agg

,order_id,total_payment,payment_type,payment_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,credit_card,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,credit_card,3
2,000229ec398224ef6ca0657da4fc703e,216.87,credit_card,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,credit_card,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,credit_card,3
...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,343.40,boleto,1
99436,fffcd46ef2263f404302a634eb57f7eb,386.53,boleto,1
99437,fffce4705a9662cd70adb13d4a31832d,116.85,credit_card,3
99438,fffe18544ffabc95dfada21779c9644f,64.71,credit_card,3


-> Get one review per order 

In [12]:
rev_agg = reviews.sort_values('review_score', ascending = False)
rev_agg = rev_agg.drop_duplicates(subset = 'order_id', keep='first')
rev_agg = rev_agg[['order_id','review_score','review_comment_message']]

In [13]:
reviews

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


In [14]:
rev_agg

,order_id,review_score,review_comment_message
99221,55d4004744368f5571d1f590031933e4,5,"Excelente mochila, entrega super rápida. Super..."
99220,22ec9f0669f784db00fa86d035cf8602,5,NaN
1,a548910a1c6147796b98fdf73dbeba33,5,NaN
2,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN
3,658677c97b385a9be170737859d3511b,5,Recebi bem antes do prazo estipulado.
...,...,...,...
57329,a63e4b82c362c488802abe5a133440aa,1,Na embalagem consta que é de 500mg porém cada ...
19,583174fbe37d3d5f0d6661be3aad1786,1,Péssimo
99223,90531360ecb1eec2a1fbb265a0db0508,1,"meu produto chegou e ja tenho que devolver, po..."
39,3c314f50bc654f3c4e317b055681dff9,1,Nada de chegar o meu pedido.


-> Translate product categories to English

In [15]:
products = products.merge(trans, on='product_category_name', how='left')
products['category_en'] = products['product_category_name_english'].fillna('Unknown')


In [16]:
products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,category_en
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares,housewares
...,...,...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0,furniture_decor,furniture_decor
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0,construction_tools_lights,construction_tools_lights
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0,bed_bath_table,bed_bath_table
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0,computers_accessories,computers_accessories


-> Merge everything into one master table

In [17]:
df = orders.merge(items_agg, on='order_id', how='left')
df = df.merge(pay_agg, on='order_id', how='left')
df = df.merge(rev_agg, on='order_id', how='left')
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(sellers[['seller_id','seller_city','seller_state']],
              on='seller_id',  how='left')
df = df.merge(products[['product_id','category_en','product_weight_g','product_length_cm']],
              on='product_id', how='left')



In [18]:
df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,revenue,freight,...,review_comment_message,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,seller_city,seller_state,category_en,product_weight_g,product_length_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,29.99,8.72,...,"Não testei o produto ainda, mas ele veio corre...",7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,maua,SP,housewares,500.0,19.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,118.70,22.76,...,Muito bom o produto.,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,belo horizonte,SP,perfumery,400.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,159.90,19.22,...,NaN,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,guariba,SP,auto,420.0,24.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,45.00,27.20,...,O produto foi exatamente o que eu esperava e e...,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,belo horizonte,MG,pet_shop,450.0,30.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,19.90,8.72,...,NaN,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,mogi das cruzes,SP,stationery,250.0,51.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,72.00,13.08,...,NaN,6359f309b166b0196dbf7ad2ac62bb5a,12209,sao jose dos campos,SP,braganca paulista,SP,health_beauty,1175.0,22.0
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,174.90,20.10,...,So uma peça que veio rachado mas tudo bem rs,da62f9e57a76d978d02ab5362c509660,11722,praia grande,SP,tupa,SP,baby,4950.0,40.0
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,205.99,65.02,...,Foi entregue antes do prazo.,737520a9aad80b3fbbdad19b66b37b30,45920,nova vicosa,BA,sao paulo,SP,home_appliances_2,13300.0,32.0
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,359.98,81.18,...,Foi entregue somente 1. Quero saber do outro p...,5097a5312c8b157bb7be58ae360ef43c,28685,japuiba,RJ,ilicinea,MG,computers_accessories,6550.0,20.0


-> Add derived columns

In [19]:
# Delivery time
df['delivery_days'] = (df['order_delivered_customer_date'] -
                       df['order_purchase_timestamp']).dt.days
df['delay_days']    = (df['order_delivered_customer_date'] -
                       df['order_estimated_delivery_date']).dt.days
df['is_late']       = (df['delay_days'] > 0).astype(int)



In [20]:
# Time columns
df['year']       = df['order_purchase_timestamp'].dt.year
df['month_num']  = df['order_purchase_timestamp'].dt.month
df['month_name'] = df['order_purchase_timestamp'].dt.strftime('%b')
df['quarter']    = 'Q' + df['order_purchase_timestamp'].dt.quarter.astype(str)
df['weekday']    = df['order_purchase_timestamp'].dt.strftime('%A')


In [21]:
# Revenue flags
df['total_order_value'] = df['revenue'] + df['freight']
df['is_returned']       = (df['order_status']=='canceled').astype(int)
df['has_review']        = df['review_score'].notna().astype(int)
df['low_rating']        = (df['review_score'] <= 2).astype(int)


In [22]:
df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,revenue,freight,...,is_late,year,month_num,month_name,quarter,weekday,total_order_value,is_returned,has_review,low_rating
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,29.99,8.72,...,0,2017,10,Oct,Q4,Monday,38.71,0,1,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,118.70,22.76,...,0,2018,7,Jul,Q3,Tuesday,141.46,0,1,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,159.90,19.22,...,0,2018,8,Aug,Q3,Wednesday,179.12,0,1,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,45.00,27.20,...,0,2017,11,Nov,Q4,Saturday,72.20,0,1,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,19.90,8.72,...,0,2018,2,Feb,Q1,Tuesday,28.62,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,72.00,13.08,...,0,2017,3,Mar,Q1,Thursday,85.08,0,1,0
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,174.90,20.10,...,0,2018,2,Feb,Q1,Tuesday,195.00,0,1,0
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,205.99,65.02,...,0,2017,8,Aug,Q3,Sunday,271.01,0,1,0
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,359.98,81.18,...,0,2018,1,Jan,Q1,Monday,441.16,0,1,1


-> Drop rows missing critical columns & export

In [23]:
df = df[df['order_status'].notna()]
df = df[df['revenue'].notna()]

In [24]:
df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,revenue,freight,...,is_late,year,month_num,month_name,quarter,weekday,total_order_value,is_returned,has_review,low_rating
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,29.99,8.72,...,0,2017,10,Oct,Q4,Monday,38.71,0,1,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,118.70,22.76,...,0,2018,7,Jul,Q3,Tuesday,141.46,0,1,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,159.90,19.22,...,0,2018,8,Aug,Q3,Wednesday,179.12,0,1,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,45.00,27.20,...,0,2017,11,Nov,Q4,Saturday,72.20,0,1,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,19.90,8.72,...,0,2018,2,Feb,Q1,Tuesday,28.62,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,72.00,13.08,...,0,2017,3,Mar,Q1,Thursday,85.08,0,1,0
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,174.90,20.10,...,0,2018,2,Feb,Q1,Tuesday,195.00,0,1,0
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,205.99,65.02,...,0,2017,8,Aug,Q3,Sunday,271.01,0,1,0
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,359.98,81.18,...,0,2018,1,Jan,Q1,Monday,441.16,0,1,1


In [25]:
df.isnull().sum()

order_id                             0
customer_id                          0
order_status                         0
order_purchase_timestamp             0
order_approved_at                   14
order_delivered_carrier_date      1009
order_delivered_customer_date     2190
order_estimated_delivery_date        0
revenue                              0
freight                              0
itemas_count                         0
avg_item_price                       0
seller_id                            0
product_id                           0
total_payment                        1
payment_type                         1
payment_installments                 1
review_score                       749
review_comment_message           58431
customer_unique_id                   0
customer_zip_code_prefix             0
customer_city                        0
customer_state                       0
seller_city                          0
seller_state                         0
category_en              

In [26]:
# ── 1. Drop the 14 rows with no approval timestamp ──────────────
df = df[df['order_approved_at'].notna()]



In [27]:
# ── 2. Drop the single payment-null row ─────────────────────────
df = df[df['total_payment'].notna()]




In [28]:
# ── 3. delivery_days / delay_days — keep null, but add a flag ───
df['is_delivered'] = df['order_delivered_customer_date'].notna().astype(int)
# delivery_days and delay_days stay null for undelivered orders



In [29]:
# ── 4. review_score — keep null, has_review flag already exists ──
# DO NOT fill review_score — use has_review=0 to exclude in DAX



In [30]:
# ── 5. review_comment — fill null with empty string ─────────────
df['review_comment_message'] = df['review_comment_message'].fillna("")



In [31]:
# ── 6. product dimensions — fill with category median ───────────
for col in ['product_weight_g', 'product_length_cm']:
    cat_med = df.groupby('category_en')[col].transform('median')
    df[col] = df[col].fillna(cat_med)

In [32]:
df.to_csv('olist_master.csv', index=False)
print(f'Master table: {len(df):,} rows x {len(df.columns)} columns')

Master table: 98,651 rows x 41 columns


In [34]:
customers.to_csv('dim_customers.csv', index=False)
sellers.to_csv('dim_sellers.csv', index=False)
products.merge(trans,on='product_category_name',how='left').to_csv(
   'dim_products.csv', index=False)
